# Tutorial 15: Cracking Equations

**Programming Design Principles / Maths for IT**

We can now represent, evaluate, and manipulate polynomials. Today we learn to *solve* them: given an equation like $3x + 7 = 22$ or $x^2 - 4x + 3 = 0$, find the values of x that make it true.

We will also learn to factor quadratics -- decomposing them into simpler pieces -- and to solve systems of equations with two unknowns.

## Solving Linear Equations

A linear equation has the form $ax + b = 0$. The solution is straightforward:

$$x = -\frac{b}{a} \quad \text{(provided } a \neq 0 \text{)}$$

In our coefficient-list convention, a linear polynomial `[b, a]` represents $ax + b$. Setting it equal to zero and solving gives $x = -b/a$.

### Your turn

Write a function `solve_linear(coeffs)` that takes `[b, a]` and returns the solution. Think about edge cases: what if $a = 0$? That means there is no x term, so it is not really a linear equation. Your function should handle this gracefully.

In [1]:
def solve_linear(coeffs):
    """Return the solution to ax + b = 0, where coeffs is [b, a].

    If a is 0 there is no x term at all, so there is either no solution
    (b is nonzero) or every x is a solution (b is also 0). We report both cases
    instead of dividing by zero.
    """
    b, a = coeffs
    if a == 0:
        if b == 0:
            return "Every x is a solution"
        return "No solution"
    return -b / a


In [2]:
print(solve_linear([7, 3]))
print(solve_linear([-15, 5]))
print(solve_linear([4, 0]))


-2.3333333333333335
3.0
No solution


## The Quadratic Formula

A quadratic equation $ax^2 + bx + c = 0$ has up to two solutions, given by:

$$x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}$$

The expression under the square root, $b^2 - 4ac$, is called the *discriminant* and tells us how many real solutions exist:

- If the discriminant is positive: two distinct real roots
- If the discriminant is zero: one repeated root
- If the discriminant is negative: no real roots (the parabola does not cross the x-axis)

### Your turn

Write a function `solve_quadratic(coeffs)` that takes `[c, b, a]` (our convention) and returns the solutions. Handle all three cases of the discriminant.

**Pseudocode:**
```
EXTRACT c, b, a from coeffs
COMPUTE discriminant = b^2 - 4*a*c
IF discriminant > 0:
    COMPUTE root1 = (-b + sqrt(discriminant)) / (2*a)
    COMPUTE root2 = (-b - sqrt(discriminant)) / (2*a)
    RETURN (root1, root2)
ELIF discriminant == 0:
    COMPUTE root = -b / (2*a)
    RETURN (root,)
ELSE:
    RETURN () or a message indicating no real roots
```

In [3]:
import math

def solve_quadratic(coeffs):
    """Return a tuple of the real solutions to ax^2 + bx + c = 0, where coeffs is [c, b, a].

    The discriminant b^2 - 4ac decides how many real roots there are: two, one
    (repeated), or none. Missing this branch is the classic bug in a quadratic
    solver -- taking the square root of a negative number without checking first.
    """
    c, b, a = coeffs
    discriminant = b ** 2 - 4 * a * c
    if discriminant > 0:
        root1 = (-b + math.sqrt(discriminant)) / (2 * a)
        root2 = (-b - math.sqrt(discriminant)) / (2 * a)
        return (root1, root2)
    elif discriminant == 0:
        root = -b / (2 * a)
        return (root,)
    else:
        return ()


In [4]:
print(solve_quadratic([3, -4, 1]))
print(solve_quadratic([1, -2, 1]))
print(solve_quadratic([5, 0, 1]))
print(solve_quadratic([-9, 0, 1]))


(3.0, 1.0)
(1.0,)
()
(3.0, -3.0)


This tutorial leans on `evaluate_poly` (to check a root) and `multiply_poly` (to check a
factorisation), both built in Tutorial 14. We bring them forward here since this notebook
starts with a fresh kernel.

In [5]:
def evaluate_poly(coeffs, x):
    """Return the value of the polynomial represented by coeffs at the given x. Carried forward from Tutorial 14."""
    result = 0
    for i in range(len(coeffs)):
        result = result + coeffs[i] * x ** i
    return result


def multiply_poly(a, b):
    """Return a new coefficient list representing the product a * b. Carried forward from Tutorial 14."""
    result_length = len(a) + len(b) - 1
    result = [0] * result_length
    for i in range(len(a)):
        for j in range(len(b)):
            result[i + j] = result[i + j] + a[i] * b[j]
    return result


### Verifying solutions

A satisfying check: if x is a root of the polynomial, then evaluating the polynomial at x should give zero (or very close to zero, allowing for floating-point imprecision).

In [6]:
# Verification pattern
coeffs = [3, -4, 1]   # x^2 - 4x + 3
roots = solve_quadratic(coeffs)
print("Roots:", roots)
for root in roots:
    value = evaluate_poly(coeffs, root)
    print("  p(" + str(root) + ") =", value)

Roots: (3.0, 1.0)
  p(3.0) = 0.0
  p(1.0) = 0.0


### Your turn

Write a function `verify_roots(coeffs, roots)` that checks whether each root really is a root by evaluating the polynomial at that point. Print PASS or FAIL for each (use a small tolerance like 0.0001 for floating-point comparison instead of exact equality).

In [7]:
def verify_roots(coeffs, roots):
    """Print PASS or FAIL for each root, checking evaluate_poly(coeffs, root) is close to zero."""
    tolerance = 0.0001
    for root in roots:
        value = evaluate_poly(coeffs, root)
        if abs(value) < tolerance:
            print("root =", root, "PASS (p(root) =", value, ")")
        else:
            print("root =", root, "FAIL (p(root) =", value, ")")


In [8]:
verify_roots([3, -4, 1], solve_quadratic([3, -4, 1]))
verify_roots([1, -2, 1], solve_quadratic([1, -2, 1]))
verify_roots([-9, 0, 1], solve_quadratic([-9, 0, 1]))


root = 3.0 PASS (p(root) = 0.0 )
root = 1.0 PASS (p(root) = 0.0 )
root = 1.0 PASS (p(root) = 0.0 )
root = 3.0 PASS (p(root) = 0.0 )
root = -3.0 PASS (p(root) = 0.0 )


## Factorisation

If we know the roots $r_1$ and $r_2$ of a quadratic $ax^2 + bx + c$, we can write it in factored form:

$$a(x - r_1)(x - r_2)$$

For example, $x^2 - 4x + 3 = (x - 1)(x - 3)$.

This is the reverse of expanding (FOIL): instead of multiplying two binomials to get a quadratic, we decompose a quadratic into two binomials.

### Your turn

Write a function `factor_quadratic(coeffs)` that returns a string showing the factored form. If the quadratic has no real roots, return a message saying it cannot be factored over the reals.

Hint: use `solve_quadratic` to find the roots, then construct the string. Be careful with the leading coefficient $a$.

In [9]:
def factor_quadratic(coeffs):
    """Return a string showing the factored form a(x - r1)(x - r2), or a message if there are no real roots."""
    c, b, a = coeffs
    roots = solve_quadratic(coeffs)

    if len(roots) == 0:
        return "Cannot be factored over the reals"

    if len(roots) == 1:
        r1 = r2 = roots[0]
    else:
        r1, r2 = roots

    def factor_string(r):
        if r >= 0:
            return "(x - " + str(r) + ")"
        return "(x + " + str(-r) + ")"

    prefix = ""
    if a != 1:
        prefix = str(a)

    return prefix + factor_string(r1) + factor_string(r2)


In [10]:
print(factor_quadratic([3, -4, 1]))
print(factor_quadratic([-6, -1, 1]))
print(factor_quadratic([5, 0, 1]))


(x - 3.0)(x - 1.0)
(x - 3.0)(x + 2.0)
Cannot be factored over the reals


### Verification by expansion

We can verify a factorisation by multiplying the factors back together and checking that we get the original polynomial. This is where `multiply_poly` from Tutorial 14 pays off:

In [11]:
# If x^2 - 4x + 3 = (x - 1)(x - 3), then:
factor1 = [-1, 1]     # (x - 1) in our convention
factor2 = [-3, 1]     # (x - 3)
product = multiply_poly(factor1, factor2)
print("Product:", product)  # should be [3, -4, 1]

Product: [3, -4, 1]


## Solving Inequalities

A linear inequality like $2x + 3 > 7$ defines a *set* of solutions rather than a single value. Solving it follows the same steps as an equation, but we need to remember: if we multiply or divide by a negative number, the inequality flips.

$$2x + 3 > 7 \implies 2x > 4 \implies x > 2$$

### Your turn

Write a function `solve_linear_inequality(a, b, c, operator)` that solves $ax + b$ [operator] $c$ where operator is one of ">", ">=", "<", "<=". Return a string describing the solution set.

Think about what happens when $a$ is negative (the inequality direction reverses).

In [12]:
def solve_linear_inequality(a, b, c, operator):
    """Return a string describing the solution set of ax + b [operator] c.

    Dividing both sides by a negative number flips the inequality, so we need
    to branch on the sign of a rather than always solving the same direction.
    """
    if a == 0:
        left = b
        if operator == ">":
            holds = left > c
        elif operator == ">=":
            holds = left >= c
        elif operator == "<":
            holds = left < c
        else:
            holds = left <= c
        return "True for all x" if holds else "No solution"

    boundary = (c - b) / a

    if a > 0:
        direction = operator
    else:
        flip = {">": "<", ">=": "<=", "<": ">", "<=": ">="}
        direction = flip[operator]

    return "x " + direction + " " + str(boundary)


In [13]:
print(solve_linear_inequality(2, 3, 7, ">"))
print(solve_linear_inequality(-3, 5, 2, "<"))
print(solve_linear_inequality(0, 5, 3, ">"))


x > 2.0
x > 1.0
True for all x


## Simultaneous Equations

Sometimes we need to find values that satisfy two equations at once. The system:

$$x + y = 10$$
$$2x - y = 5$$

has the solution $x = 5, y = 5$.

The classic approach is *elimination*: multiply the equations so that one variable cancels when we add or subtract them. For the system $a_1 x + b_1 y = c_1$ and $a_2 x + b_2 y = c_2$:

$$x = \frac{c_1 b_2 - c_2 b_1}{a_1 b_2 - a_2 b_1}, \quad y = \frac{a_1 c_2 - a_2 c_1}{a_1 b_2 - a_2 b_1}$$

The denominator $a_1 b_2 - a_2 b_1$ is called the *determinant*. If it is zero, the system has no unique solution (the lines are parallel or identical).

### Your turn

Write a function `solve_simultaneous(eq1, eq2)` where each equation is represented as `[a, b, c]` meaning $ax + by = c$. Return the values of x and y, or indicate if no unique solution exists.

In [14]:
def solve_simultaneous(eq1, eq2):
    """Return (x, y) solving a1*x + b1*y = c1 and a2*x + b2*y = c2, or a message if there is no unique solution."""
    a1, b1, c1 = eq1
    a2, b2, c2 = eq2

    determinant = a1 * b2 - a2 * b1
    if determinant == 0:
        return "No unique solution (lines are parallel or identical)"

    x = (c1 * b2 - c2 * b1) / determinant
    y = (a1 * c2 - a2 * c1) / determinant
    return (x, y)


In [15]:
print(solve_simultaneous([1, 1, 10], [2, -1, 5]))

# Solving this system by hand gives x = 2.8, y = 1.8, not x = 2, y = 3 as the
# comment above suggests -- (2, 3) satisfies 3x + 2y = 12 but not x - y = 1
# (2 - 3 = -1, not 1). Worth checking a "should give" comment against the
# equations themselves rather than trusting it outright.
print(solve_simultaneous([3, 2, 12], [1, -1, 1]))

print(solve_simultaneous([2, 4, 10], [1, 2, 5]))


(5.0, 5.0)
(2.8, 1.8)
No unique solution (lines are parallel or identical)


## Reflection

We have built equation-solving machinery from scratch: linear equations, quadratic equations (with the discriminant determining the number of solutions), factorisation, inequalities, and simultaneous equations. Each solution method is a function that takes coefficients and returns results.

The power of this approach is that we can verify everything computationally. Find a root, then evaluate the polynomial at that root to confirm it is zero. Factor a quadratic, then multiply the factors to confirm we get the original. Solve a system, then substitute back to confirm both equations hold.

Next tutorial we will work with sets -- collections where membership and relationships matter -- which is the last major topic before Skills Demo 2B.

Which type of equation did you find most satisfying to solve programmatically?

